# Regional European Daily Event Analysis: Marine Heatwave ID & Tracking using `MarEx` Regional Mode

### `MarEx` Processing Pipeline for Regional European Datasets:

1. **Morphological Pre-Processing**
    - Binary closing then opening with a disk of radius `R_fill` cells fills holes and removes features smaller than that
    - Gaps of up to `T_fill` days within an event are bridged in time
    - Objects smaller than `area_filter_absolute` cells are dropped

2. **Blob Identification**
    - Connected components are labelled on each day (8-connected, with no wrap at the domain edges)
    - Area and centroid are computed for every object

3. **Temporal Tracking**
    - Objects on consecutive days are linked where they overlap by at least `overlap_threshold` of the smaller area
    - When objects merge, the merged object keeps the parents' IDs, and each cell goes to the parent of the nearest
      cell (`nn_partitioning=True`) or of the nearest centroid

4. **Graph Reduction & Finalisation**
    - The overlap graph is reduced with `scipy.sparse.csgraph.connected_components` to one global ID per event
    - Events are mapped into ID-time space, with area, centroid, start and end for each

Every parameter is described under [Tracking & Merge/Split](https://marex.readthedocs.io/en/latest/guide/tracking.html#parameters).

`marEx.regional_tracker()` is `marEx.tracker()` with `regional_mode=True`: no periodic wrap in longitude, and `coordinate_units` given explicitly because auto-detection is unreliable over a partial coordinate range.

In [1]:
from getpass import getuser
from pathlib import Path

import xarray as xr

import marEx
import marEx.helper as hpc

2026-10-11 02:52:02 - marEx - INFO - [PID:2368099] - MarEx logging configured - Level: INFO, Mode: normal


In [2]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "regional"  # where 01_preprocess_extremes.ipynb wrote its output
temp_dir = scratch_dir / "marEx" / "examples" / "tmp"  # staging for compute_mode="streaming"

# One node: 4 workers x 16 threads x 28 GB = 64 cores, 112 GB for the workers, the rest of 128 GB for the client
n_workers, threads_per_worker, memory_limit = 4, 16, "28GB"

In [3]:
out_dir.mkdir(parents=True, exist_ok=True)

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

2026-10-11 02:52:10 - marEx.helper.cluster - INFO - [PID:2368099] - Starting local Dask cluster with 4 workers, 16 threads each


2026-10-11 02:52:10 - marEx.helper.dask_config - INFO - [PID:2368099] - Configuring Dask for HPC environment


2026-10-11 02:52:10 - marEx.helper.dask_config - INFO - [PID:2368099] - Dask temporary directory: /scratch/b/b382615/marEx/dask/tmpmqs0slfr


2026-10-11 02:52:10 - marEx.helper.dask_config - INFO - [PID:2368099] - Dask configuration completed


2026-10-11 02:52:10 - marEx.helper.cluster - INFO - [PID:2368099] - System resources: 128 physical cores, 256 logical cores, 251.6GB total memory


...

2026-10-11 02:52:10 - marEx.helper.cluster - INFO - [PID:2368099] - Starting Local cluster startup


2026-10-11 02:52:12 - marEx.helper.cluster - INFO - [PID:2368099] - Completed Local cluster startup in 2.15s


Hostname: l40016
Forward Port: l40016:8787
Dashboard Link: localhost:8787/status
2026-10-11 02:52:12 - marEx.helper.cluster - INFO - [PID:2368099] - Local cluster started successfully - Dashboard: http://127.0.0.1:8787/status


2026-10-11 02:52:12 - marEx.helper.cluster - INFO - [PID:2368099] - After cluster startup - Memory Usage - RSS: 571.6MB, Virtual: 24697.9MB, Percent: 0.2%, Available: 229771.5MB


2026-10-11 02:52:12 - marEx.helper.resources - INFO - [PID:2368099] - Cluster: 4 workers, 64 threads, memory_limit per worker [28.0] GB (112 GB total)


In [4]:
# Space stays whole because the morphology and the labelling are global in space. Time is the axis to chunk.
ds = xr.open_zarr(out_dir / "extremes_shifting_hobday.zarr", chunks={"time": 25, "lat": -1, "lon": -1})
ds

<xarray.Dataset> Size: 50GB
Dimensions:         (time: 9282, lat: 800, lon: 1300, dayofyear: 366)
Coordinates:
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
  * lat             (lat) float32 3kB 30.02 30.08 30.12 ... 69.88 69.93 69.97
  * lon             (lon) float32 5kB -19.98 -19.92 -19.88 ... 44.88 44.92 44.97
  * time            (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-0...
Data variables:
    dat_anomaly     (time, lat, lon) float32 39GB dask.array<chunksize=(25, 800, 1300), meta=np.ndarray>
    extreme_events  (time, lat, lon) bool 10GB dask.array<chunksize=(25, 800, 1300), meta=np.ndarray>
    mask            (lat, lon) bool 1MB dask.array<chunksize=(800, 1300), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 2GB dask.array<chunksize=(800, 1300, 25), meta=np.ndarray>
Attributes:
    max_anomaly:           21.51153564453125
    method_anomaly:        shifting_baseline
    method_extreme:        seasonal_percentile
    method_percentile:     approximate
    precision:             0.014341023763020834
    preprocessing_steps:   ['Rolling climatology using 15 years', 'Smoothed w...
    smooth_days:           21
    tail:                  upper
    threshold_percentile:  95
    window_days:           11
    window_spatial:        5
    window_years:          15

## Identification, Tracking, & Merging

At 25 years of daily data one int32 field is 9282 × 800 × 1300 × 4 B ≈ 39 GB, and `compute_mode="persist"` keeps several
such fields in cluster memory at once. `compute_mode="streaming"` stages them to zarr under `temp_dir` instead, so the
memory needed no longer grows with the length of the record. The staging directory outlives `run()`, so the result
is written first and the directory then released with `marEx.clear_staging`. See
[Memory Parameters](https://marex.readthedocs.io/en/latest/guide/tracking.html#compute-modes-and-larger-than-memory-tracking).

In [5]:
with monitor.stage("Track: merging & splitting"):
    tracker = marEx.regional_tracker(
        ds.extreme_events,
        ds.mask,
        coordinate_units="degrees",  # required in regional mode, where units cannot be inferred from a partial range
        grid_resolution=0.05,  # degrees, for object areas
        area_filter_absolute=600,  # drop objects smaller than 600 cells
        R_fill=16,  # fill holes up to 16 cells in radius
        T_fill=4,  # bridge gaps of up to 4 days within one event
        allow_merging=True,  # follow events through mergers and splits, recording each in `merges_ds`
        overlap_threshold=0.25,  # link objects that overlap by at least 25 % of the smaller area
        nn_partitioning=True,  # partition merged objects by nearest parent cell (False: nearest centroid, Sun et al. 2023)
        compute_mode="streaming",
        temp_dir=str(temp_dir),
        verbose=True,
    )
    extreme_events_ds, merges_ds = tracker.run(return_merges=True)
    extreme_events_ds.to_zarr(out_dir / "events_merged.zarr", mode="w")
    merges_ds.to_netcdf(out_dir / "events_merged_merges.nc", mode="w")

marEx.clear_staging(extreme_events_ds)
extreme_events_ds = xr.open_zarr(out_dir / "events_merged.zarr", chunks={})
extreme_events_ds

2026-10-11 02:52:13 - marEx - INFO - [PID:2368099] - configure_logging:177 - MarEx logging configured - Level: DEBUG, Mode: verbose


2026-10-11 02:52:13 - marEx.track.tracker - INFO - [PID:2368099] - __init__:523 - Initialising MarEx tracker


2026-10-11 02:52:13 - marEx.track.tracker - INFO - [PID:2368099] - __init__:524 - Grid type: structured


2026-10-11 02:52:13 - marEx.track.tracker - INFO - [PID:2368099] - __init__:525 - Parameters: R_fill=16, T_fill=4, area_filter_quartile=None, area_filter_absolute=600, prefilter_min_cells=None


2026-10-11 02:52:13 - marEx.track.tracker - DEBUG - [PID:2368099] - __init__:530 - Tracking options: allow_merging=True, nn_partitioning=True, overlap_threshold=0.25


...

2026-10-11 03:07:38 - marEx.track.tracker - DEBUG - [PID:2368099] - log_memory_usage:287 - Pipeline completion - Memory Usage - RSS: 1330.2MB, Virtual: 35582.7MB, Percent: 0.5%, Available: 224364.7MB


2026-10-11 03:07:38 - marEx.track.tracker - DEBUG - [PID:2368099] - run:1022 - Returning both events and merge datasets


Track: merging & splitting: 17.5 min, peak worker memory 41.0 GB of 112 GB, client 2.8 GB, spilled 0.0 GB, restarts 0


2026-10-11 03:09:44 - marEx.core.compute_mode - DEBUG - [PID:2368099] - clear_staging:119 - Cleared staging directory: /scratch/b/b382615/marEx/examples/tmp/marex_stage_2368099_6d9e96e6


<xarray.Dataset> Size: 42GB
Dimensions:       (ID: 4552, time: 9282, lat: 800, lon: 1300, component: 2,
                   sibling_ID: 14)
Coordinates:
  * ID            (ID) int32 18kB 1 2 3 4 5 6 ... 4547 4548 4549 4550 4551 4552
  * component     (component) int64 16B 0 1
  * lat           (lat) float32 3kB 30.02 30.08 30.12 ... 69.88 69.93 69.97
  * lon           (lon) float32 5kB -19.98 -19.92 -19.88 ... 44.88 44.92 44.97
  * time          (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-05-...
Dimensions without coordinates: sibling_ID
Data variables:
    ID_field      (time, lat, lon) int32 39GB dask.array<chunksize=(1, 800, 1300), meta=np.ndarray>
    area          (time, ID) float32 169MB dask.array<chunksize=(1, 4552), meta=np.ndarray>
    centroid      (component, time, ID) float32 338MB dask.array<chunksize=(1, 1, 4552), meta=np.ndarray>
    global_ID     (time, ID) int32 169MB dask.array<chunksize=(1, 4552), meta=np.ndarray>
    merge_ledger  (time, ID, sibling_ID) int32 2GB dask.array<chunksize=(1, 4552, 14), meta=np.ndarray>
    presence      (time, ID) bool 42MB dask.array<chunksize=(1, 4552), meta=np.ndarray>
    time_end      (ID) datetime64[ns] 36kB dask.array<chunksize=(4552,), meta=np.ndarray>
    time_start    (ID) datetime64[ns] 36kB dask.array<chunksize=(4552,), meta=np.ndarray>
Attributes: (12/14)
    N_events_final:              4552
    N_objects_filtered:          34736
    N_objects_prefiltered:       43719
    R_fill:                      16
    T_fill:                      4
    accepted_area_fraction:      0.9964706962724902
    ...                          ...
    area_threshold (cells):      600
    multi_parent_merges:         3101
    nn_partitioning:             1
    overlap_threshold:           0.25
    preprocessed_area_fraction:  0.6987502028915848
    total_merges:                7466

In [6]:
merges_ds

<xarray.Dataset> Size: 1MB
Dimensions:        (merge_ID: 7466, parent_idx: 14, child_idx: 14)
Dimensions without coordinates: merge_ID, parent_idx, child_idx
Data variables:
    parent_IDs     (merge_ID, parent_idx) int32 418kB 1 2 -1 -1 ... -1 -1 -1 -1
    child_IDs      (merge_ID, child_idx) int32 418kB 6 34737 -1 -1 ... -1 -1 -1
    overlap_areas  (merge_ID, parent_idx) int32 418kB 5904 2601 -1 ... -1 -1 -1
    merge_time     (merge_ID) datetime64[ns] 60kB 1997-01-02T12:00:00 ... 202...
    n_parents      (merge_ID) int8 7kB 2 2 2 2 3 3 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
    n_children     (merge_ID) int8 7kB 2 2 2 2 3 3 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
Attributes:
    fill_value:  -1

### Run Basic Tracking for Comparison

The standard method in the literature (e.g. Scannell et al.): no temporal gap filling, no merging or splitting, and no independent tracking of
the parts of a merged event.

In [7]:
with monitor.stage("Track: basic"):
    basic_tracker = marEx.regional_tracker(
        ds.extreme_events,
        ds.mask,
        coordinate_units="degrees",  # required in regional mode, where units cannot be inferred from a partial range
        area_filter_absolute=600,
        R_fill=16,
        T_fill=0,  # no temporal gap filling
        allow_merging=False,  # touching objects share one ID for all time, before and after they touch
        compute_mode="streaming",
        temp_dir=str(temp_dir),
    )
    extreme_events_basic_ds = basic_tracker.run()
    extreme_events_basic_ds.to_zarr(out_dir / "events_basic.zarr", mode="w")

marEx.clear_staging(extreme_events_basic_ds)
extreme_events_basic_ds = xr.open_zarr(out_dir / "events_basic.zarr", chunks={})
extreme_events_basic_ds

2026-10-11 03:09:44 - marEx.track.tracker - INFO - [PID:2368099] - __init__:523 - Initialising MarEx tracker


2026-10-11 03:09:44 - marEx.track.tracker - INFO - [PID:2368099] - __init__:524 - Grid type: structured


2026-10-11 03:09:44 - marEx.track.tracker - INFO - [PID:2368099] - __init__:525 - Parameters: R_fill=16, T_fill=0, area_filter_quartile=None, area_filter_absolute=600, prefilter_min_cells=None


2026-10-11 03:09:44 - marEx.track.tracker - DEBUG - [PID:2368099] - __init__:530 - Tracking options: allow_merging=False, nn_partitioning=False, overlap_threshold=0.5


2026-10-11 03:09:44 - marEx.track.tracker - DEBUG - [PID:2368099] - log_dask_info:550 - Binary input data - Dask object - Shape: (9282, 800, 1300), Chunks: ((25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25, 25..., Size: 9653280000


...

2026-10-11 06:08:12 - marEx.track.tracker - DEBUG - [PID:2368099] - log_memory_usage:287 - Pipeline completion - Memory Usage - RSS: 7852.2MB, Virtual: 42542.8MB, Percent: 3.0%, Available: 218850.2MB


2026-10-11 06:08:12 - marEx.track.tracker - DEBUG - [PID:2368099] - run:1025 - Returning events dataset only


Track: basic: 178.8 min, peak worker memory 53.3 GB of 112 GB, client 9.0 GB, spilled 0.2 GB, restarts 0


2026-10-11 06:08:30 - marEx.core.compute_mode - DEBUG - [PID:2368099] - clear_staging:119 - Cleared staging directory: /scratch/b/b382615/marEx/examples/tmp/marex_stage_2368099_fa2222b8


<xarray.Dataset> Size: 39GB
Dimensions:   (time: 9282, lat: 800, lon: 1300)
Coordinates:
  * lat       (lat) float32 3kB 30.02 30.08 30.12 30.17 ... 69.88 69.93 69.97
  * lon       (lon) float32 5kB -19.98 -19.92 -19.88 ... 44.88 44.92 44.97
  * time      (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-05-31T1...
Data variables:
    ID_field  (time, lat, lon) int32 39GB dask.array<chunksize=(1, 800, 1300), meta=np.ndarray>
Attributes:
    N_events_final:              5482
    N_objects_filtered:          34262
    N_objects_prefiltered:       44933
    R_fill:                      16
    T_fill:                      0
    accepted_area_fraction:      0.996632104022559
    allow_merging:               0
    area_filter_quartile:        0.0
    area_threshold (cells):      600
    preprocessed_area_fraction:  0.8101250625063988

## Wall Time and Memory

One row per stage, measured while the notebook ran on the cluster configured at the top. Peak worker memory is the
summed process memory the workers report to the scheduler, which is the quantity Dask compares against `memory_limit`.
The client (this kernel) is counted separately.

In [8]:
monitor.summary()

,wall (min),peak worker memory (GB),peak client memory (GB),peak spilled to disk (GB),worker restarts
stage,,,,,
Track: merging & splitting,17.50,40.97,2.83,0.0,0
Track: basic,178.76,53.30,9.02,0.2,0
total,196.26,53.30,9.02,0.2,0


In [9]:
client.close()